# 🚀 Visualisierung der Einstellungsmetadaten aus den Filmen Konrad Wolfs mit *Altair*

## Festlegen der Parameter

In [22]:
import os

# Load one key-value CSV per film whose filename contains "metadata", including subfolders.
CSV_PATH = "vis_files_shots"
CSV_PATTERN = "*metadata*.csv"
EXPORT_PATH = "export_vis_shots"

# Optional overrides, keyed by the filename before "_metadata_".
# Example: FILM_TITLES = {"Wolf_Der_kleine_Prinz": "Der kleine Prinz"}
FILM_TITLES = {}
FILM_YEARS = {}

# Plot width excludes the film labels on the left.
CHART_WIDTH = 500
FILM_LABEL_WIDTH = 210
FILM_LABEL_WRAP = 28
MIN_ROW_HEIGHT = 28

# Use your individual metadata files by default.
DOWNLOAD_TESTFILES = True
CUSTOM_ZIP_LINK = "insert link here"
CUSTOM_ARCHIVE_NAME = "custom_archive"

os.makedirs(CSV_PATH, exist_ok=True)
os.makedirs(EXPORT_PATH, exist_ok=True)

## Technische Vorbereitung

### Installation der Programmbibliotheken

In [23]:
print("Installing dependencies.")

# Install all third-party dependencies once in the active notebook kernel.
%pip install -q pandas altair==6.0.0

print("Dependencies installed.")

Installing dependencies.
Note: you may need to restart the kernel to use updated packages.
Dependencies installed.


### Import der Programmbibliotheken

In [24]:
# Handling paths, export files and ZIP archives.
from pathlib import Path
from urllib.request import urlretrieve
import webbrowser
import shutil
import zipfile

# Formatting film titles, numbers and chart expressions.
import textwrap
import math
import json

# Reading metadata tables and creating the visualisations.
import pandas as pd
import altair as alt
from IPython.display import HTML, display

try:
    from google.colab import files
    colab = True
except ImportError:
    files = None
    colab = False
    print("Using non-Colab environment.")

export_directory = Path(EXPORT_PATH).resolve()
alt.data_transformers.disable_max_rows()

# Use German number formatting and hide the English embed action menu.
EMBED_OPTIONS = {
    "actions": False,
    "renderer": "svg",
    "formatLocale": {
        "decimal": ",", "thousands": ".", "grouping": [3], "currency": ["", " €"]
    },
}
print("Dependencies imported.")

Using non-Colab environment.
Dependencies imported.


### Download des Analysematerials

In [25]:
# Download archives only when requested; reuse an existing local archive.
def download_zip(url: str, archive_name: str, target_directory: str):
    archive_path = Path(f"{archive_name}.zip")
    if not archive_path.is_file():
        print(f"Downloading {archive_path}...")
        urlretrieve(url, archive_path)
    else:
        print(f"{archive_path} already downloaded.")

    with zipfile.ZipFile(archive_path, "r") as zip_ref:
        zip_ref.extractall(Path(target_directory))
    print(f"Extracted {archive_path} to {target_directory}.")


if DOWNLOAD_TESTFILES:
    download_zip(
        "https://owncloud.gwdg.de/index.php/s/21pPnALEiJ2O4Ar/download",
        "visualisation_shot_testfiles", CSV_PATH,
    )
else:
    print("No testfile download requested.")

if CUSTOM_ZIP_LINK != "insert link here":
    download_zip(CUSTOM_ZIP_LINK, CUSTOM_ARCHIVE_NAME, CSV_PATH)

visualisation_shot_testfiles.zip already downloaded.
Extracted visualisation_shot_testfiles.zip to vis_files_shots.


## Einlesen der Metadaten und Erstellung der Visualisierung

### Import und Transformation der Metadaten

In [26]:
# Read the headerless key-value files exported by the shot detection notebook.
metadata_files = sorted(
    path for path in Path(CSV_PATH).rglob(CSV_PATTERN) if path.is_file()
)
if not metadata_files:
    raise FileNotFoundError(
        f"Keine Metadatendateien für {CSV_PATTERN!r} in {CSV_PATH!r} gefunden. "
        "Legen Sie dort eine Metadatendatei pro Film ab."
    )

time_metrics = [
    ("mean_duration_s", "Mittlere Einstellungslänge (Sekunden)"),
    ("median_duration_s", "Mediane Einstellungslänge (Sekunden)"),
    ("max_duration_s", "Längste Einstellung (Sekunden)"),
    ("min_duration_s", "Kürzeste Einstellung (Sekunden)"),
    ("runtime_s", "Filmlaufzeit (Minuten)"),
]
shot_metric_label = "Anzahl der Einstellungen"
metric_order = [label for _, label in time_metrics] + [shot_metric_label]


def format_number(value):
    return f"{value:,.2f}".rstrip("0").rstrip(".").replace(
        ",", "TEMP"
    ).replace(".", ",").replace("TEMP", ".")


def format_duration(seconds):
    seconds = int(round(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


film_records = []
seen_ids = set()
seen_titles = set()
for csv_file in metadata_files:
    film_id = csv_file.stem.rsplit("_metadata_", 1)[0]
    if film_id in seen_ids:
        raise ValueError(
            f"Mehrere Metadatendateien für {film_id!r} gefunden. "
            "Verwenden Sie eine Datei pro Film oder schränken Sie CSV_PATTERN ein."
        )
    seen_ids.add(film_id)

    table = pd.read_csv(csv_file, header=None, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if table.shape[1] != 2:
        raise ValueError(f"{csv_file.name}: Erwartet werden zwei Spalten ohne Kopfzeile.")
    keys = table.iloc[:, 0].str.strip()
    if keys.duplicated().any():
        raise ValueError(f"{csv_file.name}: Metadatenschlüssel sind mehrfach vorhanden.")
    metadata = dict(zip(keys, table.iloc[:, 1].str.strip()))

    required = [key for key, _ in time_metrics if key != "runtime_s"]
    required += ["shot_count", "duration_hh_mm_ss"]
    missing = [key for key in required if not metadata.get(key)]
    if missing:
        raise ValueError(f"{csv_file.name}: Fehlende Metadaten: {', '.join(missing)}.")

    default_title = film_id.removeprefix("Wolf_").replace("_", " ")
    title = str(FILM_TITLES.get(film_id, metadata.get("title") or default_title)).strip()
    if not title or title in seen_titles:
        raise ValueError(f"{csv_file.name}: Bitte einen eindeutigen Filmtitel in FILM_TITLES festlegen.")
    seen_titles.add(title)

    values = {}
    for key in required:
        if key == "duration_hh_mm_ss":
            try:
                value = pd.to_timedelta(metadata[key]).total_seconds()
            except (ValueError, TypeError) as exc:
                raise ValueError(f"{csv_file.name}: Ungültige Filmlaufzeit {metadata[key]!r}.") from exc
            key = "runtime_s"
        else:
            try:
                value = float(metadata[key])
            except ValueError as exc:
                raise ValueError(f"{csv_file.name}: Ungültiger Zahlenwert für {key}.") from exc
        if not math.isfinite(value) or value < 0:
            raise ValueError(f"{csv_file.name}: {key} muss eine endliche, nicht negative Zahl sein.")
        values[key] = value
    if not values["shot_count"].is_integer():
        raise ValueError(f"{csv_file.name}: Die Einstellungsanzahl muss ganzzahlig sein.")

    year_value = FILM_YEARS.get(film_id, metadata.get("year", ""))
    if year_value is None or str(year_value).strip() == "":
        year = None
    else:
        try:
            numeric_year = float(year_value)
        except (ValueError, TypeError) as exc:
            raise ValueError(f"{csv_file.name}: Ungültiges Erscheinungsjahr {year_value!r}.") from exc
        if not math.isfinite(numeric_year) or not numeric_year.is_integer():
            raise ValueError(f"{csv_file.name}: Das Erscheinungsjahr muss ganzzahlig sein.")
        year = int(numeric_year)

    film_records.append({"film_id": film_id, "title": title, "year": year, "values": values})
    print(f"Loaded metadata for {title} from {csv_file.name}.")

# Rank films alphabetically.
alphabetical = sorted(film_records, key=lambda film: film["title"].casefold())
alpha_rank = {film["film_id"]: i for i, film in enumerate(alphabetical)}


rows = []
for film in film_records:
    title_lines = textwrap.wrap(
        film["title"], width=FILM_LABEL_WRAP, break_long_words=True, break_on_hyphens=False
    )
    common = {
        "film_id": film["film_id"], "title": film["title"],
        "film_label": "\n".join(title_lines),
        "release_year_display": str(film["year"]) if film["year"] is not None else "Nicht angegeben",
        "alpha_rank": alpha_rank[film["film_id"]],
    }
    for key, label in time_metrics:
        seconds = film["values"][key]
        runtime = key == "runtime_s"
        rows.append({
            **common, "metric": label,
            "value": seconds / 60 if runtime else seconds,
            "display_value": format_duration(seconds) if runtime else f"{format_number(seconds)} s",
        })
    count = int(film["values"]["shot_count"])
    rows.append({
        **common, "metric": shot_metric_label, "value": count,
        "display_value": f"{count:,}".replace(",", "."),
    })

df_long = pd.DataFrame(rows)
display(df_long[["title", "metric", "display_value", "release_year_display"]].rename(columns={
    "title": "Film", "metric": "Messgröße", "display_value": "Wert",
    "release_year_display": "Erscheinungsjahr",
}).head(12))

Loaded metadata for Der geteilte Himmel from Wolf_Der_geteilte_Himmel_metadata_0.5.csv.
Loaded metadata for Der kleine Prinz from Wolf_Der_kleine_Prinz_metadata_0.5.csv.
Loaded metadata for Der nackte Mann auf dem Sportplatz from Wolf_Der_nackte_Mann_auf_dem_Sportplatz_metadata_0.5.csv.
Loaded metadata for Einmal ist keinmal from Wolf_Einmal_ist_keinmal_metadata_0.5.csv.
Loaded metadata for Genesung from Wolf_Genesung_metadata_0.5.csv.
Loaded metadata for Goya from Wolf_Goya_metadata_0.5.csv.
Loaded metadata for Ich war neunzehn from Wolf_Ich_war_neunzehn_metadata_0.5.csv.
Loaded metadata for Leute mit Flügeln from Wolf_Leute_mit_Flügeln_metadata_0.5.csv.
Loaded metadata for Lissy from Wolf_Lissy_metadata_0.5.csv.
Loaded metadata for Mama ich lebe from Wolf_Mama_ich_lebe_metadata_0.5.csv.
Loaded metadata for Professor Mamlock from Wolf_Professor_Mamlock_metadata_0.5.csv.
Loaded metadata for Solo Sunny from Wolf_Solo_Sunny_metadata_0.5.csv.
Loaded metadata for Sonnensucher from Wolf_S

,Film,Messgröße,Wert,Erscheinungsjahr
0,Der geteilte Himmel,Mittlere Einstellungslänge (Sekunden),"11,31 s",Nicht angegeben
1,Der geteilte Himmel,Mediane Einstellungslänge (Sekunden),"7,3 s",Nicht angegeben
2,Der geteilte Himmel,Längste Einstellung (Sekunden),"96,36 s",Nicht angegeben
3,Der geteilte Himmel,Kürzeste Einstellung (Sekunden),"0,64 s",Nicht angegeben
4,Der geteilte Himmel,Filmlaufzeit (Minuten),01:48:54,Nicht angegeben
5,Der geteilte Himmel,Anzahl der Einstellungen,578,Nicht angegeben
6,Der kleine Prinz,Mittlere Einstellungslänge (Sekunden),"11,15 s",Nicht angegeben
7,Der kleine Prinz,Mediane Einstellungslänge (Sekunden),"6,2 s",Nicht angegeben
8,Der kleine Prinz,Längste Einstellung (Sekunden),"130,6 s",Nicht angegeben
9,Der kleine Prinz,Kürzeste Einstellung (Sekunden),"0,24 s",Nicht angegeben


### Erstellung der interaktiven Visualisierung

In [27]:
color_scale = alt.Scale(
    domain=metric_order,
    range=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"],
)
sort_mode = alt.param(
    name="sort_mode",
    value="Werte",
    bind=alt.binding_radio(
        options=["Werte", "Alphabetisch"],
        name="Sortierung: ",
    ),
)
hover = alt.selection_point(
    name="film_hover", fields=["film_id"],
    on="pointerover", clear="pointerout", empty=False,
)

# Reserve room for the value label at the end of the longest bar in each panel.
metric_limits = {
    metric: max(float(group["value"].max()), 0.1) * 1.22
    for metric, group in df_long.groupby("metric")
}
limit_expression = alt.ExprRef(
    expr=f"({json.dumps(metric_limits, ensure_ascii=False)})[parent.metric]"
)
max_label_lines = int(df_long["film_label"].str.count("\n").max()) + 1
row_height = max(MIN_ROW_HEIGHT, max_label_lines * 14 + 10)

y_encoding = alt.Y(
    "film_label:N",
    title=None,
    sort=alt.EncodingSortField(field="dynamic_sort", op="max", order="ascending"),
    axis=alt.Axis(
        labelAngle=0, labelLimit=FILM_LABEL_WIDTH,
        labelExpr='split(datum.label, "\\n")',
        labelLineHeight=14, labelPadding=10, ticks=False, domain=False,
    ),
)
base = (
    alt.Chart(df_long)
    .transform_calculate(
        dynamic_sort=(
            "sort_mode === 'Alphabetisch' ? datum.alpha_rank "
            ": datum.metric === 'Kürzeste Einstellung (Sekunden)' ? datum.value : -datum.value"
        )
    )
    .encode(
        x=alt.X(
            "value:Q", title=None,
            scale=alt.Scale(domainMin=0, domainMax=limit_expression, nice=False),
            axis=alt.Axis(tickCount=6, format=",.2~f", grid=True),
        ),
        y=y_encoding,
        color=alt.Color("metric:N", scale=color_scale, legend=None),
        tooltip=[
            alt.Tooltip("title:N", title="Film"),
            alt.Tooltip("release_year_display:N", title="Erscheinungsjahr"),
            alt.Tooltip("metric:N", title="Messgröße"),
            alt.Tooltip("display_value:N", title="Wert"),
        ],
    )
    .properties(width=CHART_WIDTH, height=alt.Step(row_height))
)
bars = (
    base.mark_bar(size=min(22, row_height * 0.65))
    .encode(opacity=alt.condition(hover, alt.value(1.0), alt.value(0.6)))
    .add_params(hover)
)
labels = (
    base.mark_text(align="left", baseline="middle", dx=7, fontSize=11)
    .encode(
        color=alt.value("#333333"),
        text=alt.condition(hover, alt.Text("display_value:N"), alt.value("")),
    )
)
chart = (
    (bars + labels)
    .add_params(sort_mode)
    .facet(
        row=alt.Row(
            "metric:N", sort=metric_order, title=None,
            header=alt.Header(
                labelAngle=0, labelOrient="top", labelAlign="left",
                labelAnchor="start", labelPadding=12,
                labelFontSize=14, labelFontWeight="bold",
            ),
        ),
        spacing=28,
    )
    .resolve_scale(x="independent", y="independent")
    .properties(
        title="Einstellungslängen und Einstellungsanzahl im Filmvergleich",
        padding={"top": 5, "bottom": 5, "left": 5, "right": 5},
    )
    .configure_view(stroke=None)
    .configure_title(fontSize=17, anchor="start", offset=18)
    .configure_axis(labelFontSize=11, titleFontSize=12, gridColor="#E4E4E4")
    .configure_axisY(grid=False)
)
# Keep the notebook output in normal document flow in Jupyter Book.
chart_fragment = chart.to_html(
    fullhtml=False, output_div="einstellungsmetadaten", embed_options=EMBED_OPTIONS,
    template="universal",
)
chart_styles = """<style>
#einstellungsmetadaten-ausgabe {
    margin: 0; padding: 0; height: auto; min-height: 0;
    white-space: normal; line-height: normal;
}
#einstellungsmetadaten-ausgabe #einstellungsmetadaten.vega-embed {
    display: block; width: 100%; height: auto; min-height: 0;
    margin: 0; padding: 0;
}
#einstellungsmetadaten > svg {
    display: block; max-width: 100%; height: auto;
}
#einstellungsmetadaten .vega-bindings {
    margin: 8px 0 0; padding: 0;
}
</style>"""
chart_html = (
    chart_styles
    + '<div id="einstellungsmetadaten-ausgabe">'
    + chart_fragment
    + '</div>'
)
display(HTML(chart_html))

## Export der Ergebnisse

### Export der Visualisierung als `.html`-Datei

In [28]:
# Export the same German interface in local and Google Colab environments.
output_html = export_directory / "shot_meta.html"
output_html.write_text(
    '<!DOCTYPE html><html lang="de"><head><meta charset="UTF-8">'
    '<meta name="viewport" content="width=device-width, initial-scale=1">'
    '<title>Einstellungsmetadaten</title></head>'
    '<body style="margin:0">' + chart_html + '</body></html>',
    encoding="utf-8",
)
print(f"Saved HTML visualization to: {output_html}")

if not colab:
    webbrowser.open_new_tab(output_html.as_uri())

Saved HTML visualization to: /Users/winter/Documents/MyCode/Quadriga/Bewegtes-Bild-Fallstudie-4/auswertung/export_vis_shots/shot_meta.html


### Herunterladen des *Google Colab* Export-Ordners

In [29]:
if colab:
    archive_base = export_directory.parent / f"{export_directory.name}_ergebnisse"
    zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(export_directory))
    files.download(zip_path)
    print(f"Downloaded {Path(zip_path).name}.")
else:
    print("An additional download is only required in Google Colab.")

An additional download is only required in Google Colab.
